# Model V1

In [ ]:
import sys
sys.path.append('../data_extraction')

import numpy as np
import mlflow
import keras
import tensorflow as tf

from keras.layers import Conv2D, MaxPooling2D, Conv2DTranspose, Concatenate, Dropout
from keras.regularizers import l2

from LRO_data_class import getSplitIndices, getNormalisedBatch, getAugmentedBatch, patchGenerator, stepsPerEpoch

print(tf.config.list_physical_devices('GPU'))Íﬁﬁﬁ›

In [10]:
train_idx, val_idx, test_idx = getSplitIndices()
print(f'train: {len(train_idx)}  val: {len(val_idx)}  test: {len(test_idx)}')

wac_train, dem_train, mask_train = getAugmentedBatch(batch_num=0)

# validation/test: normalised only, no augmentation
wac_val, dem_val, mask_val = getNormalisedBatch(batch_num=0)

# wac  (1000, 256, 256) float32 [0, 1]  - per-patch percentile normalised
# dem  (1000, 256, 256) float32 [0, 1]  - per-patch percentile normalised
# mask (1000, 256, 256) uint8   {0, 1}  - 1 px ring masks, ~37:1 imbalance
print(wac_train.shape, wac_train.dtype, wac_train.min(), wac_train.max())

train: 136109  val: 34699  test: 35059
(1000, 256, 256) float32 0.0 1.0


## Hyperparameters

In [ ]:
# all hyperparameters in one place — change here and MLflow logs them automatically

# [NOTES]: 
    # - plain BCE collapses to the 2.6% background base rate (loss plateaus at ~0.134 = base-rate BCE)
    # 37:1 imbalance needs weighted BCE or Dice. 
    # DeepMoon used unweighted BCE.\
    # this model learns to predict "no crater" everywhere and stops improving, 
    # because 97% of pixels really are background
    # WEIGHTED LOSS

params = {
    'dim': 256,
    'channels': 'both',                 # 'both' | 'wac' | 'dem'
    'input_channels': 2,                # 2 for both, 1 for ablations
    'n_filters': 32,                    # DeepMoon used 112
    'FL': 3,                            # kernel size
    'init': 'he_normal',
    'lmbda': 1e-6,                      # L2 strength
    'dropout': 0.15,
    'learning_rate': 0.0001,
    'batch_size': 8,
    'epochs': 50,
    'loss': 'binary_crossentropy',      # between BCE and weighted BCE
    'model': 'U-Net-v1',
}

## Model Architecture

In [12]:
img_input = keras.Input(shape=(params['dim'], params['dim'], params['input_channels']))

# Encoder1
print("-----E1-----")
a1 = Conv2D(
    params['n_filters'], 
    params['FL'], 
    activation='relu', 
    kernel_initializer=params['init'], 
    kernel_regularizer=l2(params['lmbda']), 
    padding='same'
)(img_input)

a1 = Conv2D(
    params['n_filters'], 
    params['FL'], 
    activation='relu', 
    kernel_initializer=params['init'], 
    kernel_regularizer=l2(params['lmbda']), 
    padding='same'
)(a1)

a1P = MaxPooling2D((2, 2), strides=(2, 2))(a1)

# Encoder2
print("-----E2-----")
a2 = Conv2D(
    params['n_filters'] * 2, 
    params['FL'], 
    activation='relu', 
    kernel_initializer=params['init'], 
    kernel_regularizer=l2(params['lmbda']), 
    padding='same'
)(a1P)

a2 = Conv2D(
    params['n_filters'] * 2, 
    params['FL'], 
    activation='relu', 
    kernel_initializer=params['init'], 
    kernel_regularizer=l2(params['lmbda']), 
    padding='same'
)(a2)

a2P = MaxPooling2D((2, 2), strides=(2, 2))(a2)

# Encoder3
print("-----E3-----")
a3 = Conv2D(
    params['n_filters'] * 4, 
    params['FL'], 
    activation='relu', 
    kernel_initializer=params['init'], 
    kernel_regularizer=l2(params['lmbda']), 
    padding='same'
)(a2P)

a3 = Conv2D(
    params['n_filters'] * 4, 
    params['FL'], 
    activation='relu', 
    kernel_initializer=params['init'], 
    kernel_regularizer=l2(params['lmbda']), 
    padding='same'
)(a3)

a3P = MaxPooling2D((2, 2), strides=(2, 2))(a3)

# Encoder4
print("-----E4-----")
a4 = Conv2D(
    params['n_filters'] * 8, 
    params['FL'], 
    activation='relu', 
    kernel_initializer=params['init'], 
    kernel_regularizer=l2(params['lmbda']), 
    padding='same'
)(a3P)

a4 = Conv2D(
    params['n_filters'] * 8, 
    params['FL'], 
    activation='relu', 
    kernel_initializer=params['init'], 
    kernel_regularizer=l2(params['lmbda']), 
    padding='same'
)(a4)

a4P = MaxPooling2D((2, 2), strides=(2, 2))(a4)

u = Conv2D(
    params['n_filters'] * 16, 
    params['FL'], 
    activation='relu', 
    kernel_initializer=params['init'], 
    kernel_regularizer=l2(params['lmbda']), 
    padding='same'
)(a4P)

u = Conv2D(
    params['n_filters'] * 16, 
    params['FL'], 
    activation='relu', 
    kernel_initializer=params['init'], 
    kernel_regularizer=l2(params['lmbda']), 
    padding='same'
)(u)



# Decoder1
d1CT = Conv2DTranspose(params['n_filters']*8, kernel_size=2, strides=2, padding='same')(u)
d1c = Concatenate()([d1CT, a4])
x1 = Dropout(params['dropout'])(d1c)

print("-----D1-----")
d1 = Conv2D(
    params['n_filters'] * 8, 
    params['FL'], 
    activation='relu', 
    kernel_initializer=params['init'], 
    kernel_regularizer=l2(params['lmbda']), 
    padding='same'
)(x1)

d1 = Conv2D(
    params['n_filters'] * 8, 
    params['FL'], 
    activation='relu', 
    kernel_initializer=params['init'], 
    kernel_regularizer=l2(params['lmbda']), 
    padding='same'
)(d1)

# Decoder2
print("-----D2-----")
d2CT = Conv2DTranspose(params['n_filters']*4, kernel_size=2, strides=2, padding='same')(d1)
d2c = Concatenate()([d2CT, a3])
x2 = Dropout(params['dropout'])(d2c)

d2 = Conv2D(
    params['n_filters'] * 4, 
    params['FL'], 
    activation='relu', 
    kernel_initializer=params['init'], 
    kernel_regularizer=l2(params['lmbda']), 
    padding='same'
)(x2)

d2 = Conv2D(
    params['n_filters'] * 4, 
    params['FL'], 
    activation='relu', 
    kernel_initializer=params['init'], 
    kernel_regularizer=l2(params['lmbda']), 
    padding='same'
)(d2)

# Decoder3
print("-----D3-----")
d3CT = Conv2DTranspose(params['n_filters']*2, kernel_size=2, strides=2, padding='same')(d2)
d3c = Concatenate()([d3CT, a2])
x3 = Dropout(params['dropout'])(d3c)

d3 = Conv2D(
    params['n_filters'] * 2, 
    params['FL'], 
    activation='relu', 
    kernel_initializer=params['init'], 
    kernel_regularizer=l2(params['lmbda']), 
    padding='same'
)(x3)

d3 = Conv2D(
    params['n_filters'] * 2, 
    params['FL'], 
    activation='relu', 
    kernel_initializer=params['init'], 
    kernel_regularizer=l2(params['lmbda']), 
    padding='same'
)(d3)

# Decoder4
print("-----D4-----")
d4CT = Conv2DTranspose(params['n_filters'], kernel_size=2, strides=2, padding='same')(d3)
d4c = Concatenate()([d4CT, a1])
x4 = Dropout(params['dropout'])(d4c)

d4 = Conv2D(
    params['n_filters'], 
    params['FL'], 
    activation='relu', 
    kernel_initializer=params['init'], 
    kernel_regularizer=l2(params['lmbda']), 
    padding='same'
)(x4)

d4 = Conv2D(
    params['n_filters'], 
    params['FL'], 
    activation='relu', 
    kernel_initializer=params['init'], 
    kernel_regularizer=l2(params['lmbda']), 
    padding='same'
)(d4)

# Output layer
output = Conv2D(1, 1, activation='sigmoid')(d4)
model = keras.Model(img_input, output)
model.summary()

-----E1-----
-----E2-----
-----E3-----
-----E4-----
-----D1-----
-----D2-----
-----D3-----
-----D4-----


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_1       │ (None, 256, 256,  │          0 │ -                 │
│ (InputLayer)        │ 2)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_10 (Conv2D)  │ (None, 256, 256,  │        608 │ input_layer_1[0]… │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_11 (Conv2D)  │ (None, 256, 256,  │      9,248 │ conv2d_10[0][0]   │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_4     │ (None, 128, 128,  │          0 │ conv2d_11[0][0]   │
│ (MaxPooling2D)      │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_12 (Conv2D)  │ (None, 128, 128,  │     18,496 │ max_pooling2d_4[… │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_13 (Conv2D)  │ (None, 128, 128,  │     36,928 │ conv2d_12[0][0]   │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_5     │ (None, 64, 64,    │          0 │ conv2d_13[0][0]   │
│ (MaxPooling2D)      │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_14 (Conv2D)  │ (None, 64, 64,    │     73,856 │ max_pooling2d_5[… │
│                     │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_15 (Conv2D)  │ (None, 64, 64,    │    147,584 │ conv2d_14[0][0]   │
│                     │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_6     │ (None, 32, 32,    │          0 │ conv2d_15[0][0]   │
│ (MaxPooling2D)      │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_16 (Conv2D)  │ (None, 32, 32,    │    295,168 │ max_pooling2d_6[… │
│                     │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_17 (Conv2D)  │ (None, 32, 32,    │    590,080 │ conv2d_16[0][0]   │
│                     │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_7     │ (None, 16, 16,    │          0 │ conv2d_17[0][0]   │
│ (MaxPooling2D)      │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_18 (Conv2D)  │ (None, 16, 16,    │  1,180,160 │ max_pooling2d_7[… │
│                     │ 512)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_19 (Conv2D)  │ (None, 16, 16,    │  2,359,808 │ conv2d_18[0][0]   │
│                     │ 512)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_transpose_1  │ (None, 32, 32,    │    524,544 │ conv2d_19[0][0]   │
│ (Conv2DTranspose)   │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate_1       │ (None, 32, 32,    │          0 │ conv2d_transpose

 Total params: 7,759,809 (29.60 MB)

 Trainable params: 7,759,809 (29.60 MB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
model.compile(
    optimizer=keras.optimizers.Adam(params['learning_rate']),
    loss=keras.losses.BinaryFocalCrossentropy(apply_class_balancing=True, alpha=0.25, gamma=2.0)
)

# smoke test only - 8 patches, no augmentation, must drive loss toward 0.
# if it plateaus, the bug is upstream in the pipeline, not the model.
model.fit(
    patchGenerator(train_idx[:8], channels=params['channels'],batch_size=params['batch_size'], augment_data=False),
    steps_per_epoch=10,
    epochs=params['epochs'],
)

/Users/none/miniconda3/envs/lunar_lro/lib/python3.11/site-packages/keras/src/trainers/epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a generator. The generator is expected to yield already-shuffled data.
  self.data_adapter = data_adapters.get_data_adapter(


Epoch 1/50
1/1 ━━━━━━━━━━━━━━━━━━━━ 11s 11s/step - loss: 0.8667
Epoch 2/50
1/1 ━━━━━━━━━━━━━━━━━━━━ 5s 5s/step - loss: 0.7717
Epoch 3/50
1/1 ━━━━━━━━━━━━━━━━━━━━ 9s 9s/step - loss: 0.6972
Epoch 4/50
1/1 ━━━━━━━━━━━━━━━━━━━━ 36s 36s/step - loss: 0.6339
Epoch 5/50
1/1 ━━━━━━━━━━━━━━━━━━━━ 8s 8s/step - loss: 0.5788
Epoch 6/50
1/1 ━━━━━━━━━━━━━━━━━━━━ 11s 11s/step - loss: 0.5279
Epoch 7/50
1/1 ━━━━━━━━━━━━━━━━━━━━ 14s 14s/step - loss: 0.4810
Epoch 8/50
1/1 ━━━━━━━━━━━━━━━━━━━━ 11s 11s/step - loss: 0.4330
Epoch 9/50
1/1 ━━━━━━━━━━━━━━━━━━━━ 10s 10s/step - loss: 0.3848
Epoch 10/50
1/1 ━━━━━━━━━━━━━━━━━━━━ 13s 13s/step - loss: 0.3329
Epoch 11/50
1/1 ━━━━━━━━━━━━━━━━━━━━ 15s 15s/step - loss: 0.2791
Epoch 12/50
1/1 ━━━━━━━━━━━━━━━━━━━━ 11s 11s/step - loss: 0.2283
Epoch 13/50
1/1 ━━━━━━━━━━━━━━━━━━━━ 18s 18s/step - loss: 0.1937
Epoch 14/50
1/1 ━━━━━━━━━━━━━━━━━━━━ 17s 17s/step - loss: 0.1757
Epoch 15/50
1/1 ━━━━━━━━━━━━━━━━━━━━ 12s 12s/step - loss: 0.1730
Epoch 16/50
1/1 ━━━━━━━━━━━━━━━━━━━━ 16s

## Training

In [ ]:
# MLflow tracks every training run — hyperparameters, metrics, and the model itself
# run `mlflow ui` in the terminal and localhost:5000
# each run is logged separately to compare experiments side by side

# [source]: https://mlflow.org/docs/latest/python_api/mlflow.keras.html
# [example source]: https://github.com/mlflow/mlflow/blob/master/examples/keras/train.py

mlflow.set_experiment('lunar-crater-detection')

with mlflow.start_run(run_name='fusion-wac-dem'):
    mlflow.log_params(params)

    history = model.fit(
        patchGenerator(train_idx, channels=params['channels'],batch_size=params['batch_size']),
        steps_per_epoch=stepsPerEpoch(train_idx, params['batch_size']),
        validation_data=patchGenerator(val_idx, channels=params['channels'],batch_size=params['batch_size'], augment_data=False),
        validation_steps=stepsPerEpoch(val_idx, params['batch_size']),
        epochs=params['epochs'],
    )

    for epoch, (tl, vl) in enumerate(zip(history.history['loss'],
                                         history.history['val_loss'])):
        mlflow.log_metric('train_loss', tl, step=epoch)
        mlflow.log_metric('val_loss', vl, step=epoch)

    mlflow.keras.log_model(model, 'model')

ValueError: Unrecognized data type: x=Ellipsis (of type <class 'ellipsis'>)

## Evaluation